# 使用 Claude API 进行提示词缓存

提示词缓存让你能够在提示词中存储并复用上下文，从而在重复性任务中把时延降低 2 倍以上、成本最多降低 90%。

启用提示词缓存有两种方式：

- **自动缓存**（推荐）：在请求的顶层添加一个 `cache_control` 字段。系统会自动为你管理缓存断点。
- **显式缓存断点**：把 `cache_control` 放在各个内容块上，以便精确控制究竟哪些内容被缓存。

本示例演示两种方式，先从更简单的自动方式开始。

## 前置条件

In [1]:
%pip install --upgrade 'anthropic>=0.83.0' bs4 requests python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time

import anthropic
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

# Unique prefix to ensure we don't hit a stale cache from a previous run
TIMESTAMP = int(time.time())

我们获取《傲慢与偏见》全文（约 18.7 万词元）作为大上下文。

In [3]:
def fetch_article_content(url):
    response = requests.get(url, timeout=30)
    soup = BeautifulSoup(response.content, "html.parser")

    for script in soup(["script", "style"]):
        script.decompose()

    text = soup.get_text()
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = "\n".join(chunk for chunk in chunks if chunk)

    return text


book_url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
book_content = fetch_article_content(book_url)

print(f"Fetched {len(book_content)} characters from the book.")
print("First 500 characters:")
print(book_content[:500])

Fetched 737526 characters from the book.
First 500 characters:
The Project Gutenberg eBook of Pride and Prejudice
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.
Title:


我们另外定义一个用于打印用量统计的小辅助函数：

In [4]:
def print_usage(response, elapsed):
    """Print token usage and timing for an API response."""
    usage = response.usage
    cache_create = getattr(usage, "cache_creation_input_tokens", 0)
    cache_read = getattr(usage, "cache_read_input_tokens", 0)

    print(f"  Time:                {elapsed:.2f}s")
    print(f"  Input tokens:        {usage.input_tokens}")
    print(f"  Output tokens:       {usage.output_tokens}")
    if cache_create:
        print(f"  Cache write tokens:  {cache_create}")
    if cache_read:
        print(f"  Cache read tokens:   {cache_read}")

---
## 示例 1：自动缓存（单轮）

自动缓存是最简单的入门方式。在 `messages.create()` 调用的**顶层**添加 `cache_control={"type": "ephemeral"}`，其余部分由系统处理——自动把缓存断点放在最后一个可缓存的块上。

我们比较三种情形：
1. **不缓存**——基线
2. **第一次缓存调用**——创建缓存条目（时延与基线相近）
3. **第二次缓存调用**——从缓存读取（显著提速）

### 基线：不缓存

In [5]:
start = time.time()
baseline_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
baseline_time = time.time() - start

print(f"Response: {baseline_response.content[0].text}")
print_usage(baseline_response, baseline_time)

Response: Pride and Prejudice
  Time:                4.89s
  Input tokens:        187364
  Output tokens:       8


### 使用自动缓存的第一次调用（缓存写入）

唯一的改动是顶层的 `cache_control` 参数。第一次调用会写入缓存，因此时延与基线相近。

In [6]:
start = time.time()
write_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    cache_control={"type": "ephemeral"},  # <-- one-line change
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
write_time = time.time() - start

print(f"Response: {write_response.content[0].text}")
print_usage(write_response, write_time)

Response: Pride and Prejudice
  Time:                4.28s
  Input tokens:        3
  Output tokens:       8
  Cache write tokens:  187361


### 使用自动缓存的第二次调用（缓存命中）

同样的请求再来一次。这次复用了已缓存的前缀，因此你应当能看到明显的提速。

In [7]:
start = time.time()
hit_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    cache_control={"type": "ephemeral"},
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
hit_time = time.time() - start

print(f"Response: {hit_response.content[0].text}")
print_usage(hit_response, hit_time)

print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print(f"No caching:     {baseline_time:.2f}s")
print(f"Cache write:    {write_time:.2f}s")
print(f"Cache hit:      {hit_time:.2f}s")
print(f"Speedup:        {baseline_time / hit_time:.1f}x")

Response: Pride and Prejudice
  Time:                1.48s
  Input tokens:        3
  Output tokens:       8
  Cache read tokens:   187361

COMPARISON
No caching:     4.89s
Cache write:    4.28s
Cache hit:      1.48s
Speedup:        3.3x


---
## 示例 2：多轮对话中的自动缓存

自动缓存在多轮对话中表现出色——随着对话增长，缓存断点会**自动向前移动**——你不需要自己管理任何断点。

| 请求 | 缓存行为 |
|---------|----------------|
| 请求 1 | 缓存 System + User：A（写入） |
| 请求 2 | 从缓存读取 System + User：A；把 Asst：B + User：C 写入缓存 |
| 请求 3 | 从缓存读取 System 到 User：C；把 Asst：D + User：E 写入缓存 |

In [8]:
system_message = f"{TIMESTAMP} <file_contents> {book_content} </file_contents>"

questions = [
    "What is the title of this novel?",
    "Who are Mr. and Mrs. Bennet?",
    "What is Netherfield Park?",
    "What is the main theme of this novel?",
]

conversation = []

for i, question in enumerate(questions, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}: {question}")
    print("=" * 50)

    conversation.append({"role": "user", "content": question})

    start = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        cache_control={"type": "ephemeral"},  # automatic caching
        system=system_message,
        messages=conversation,
    )
    elapsed = time.time() - start

    assistant_reply = response.content[0].text
    conversation.append({"role": "assistant", "content": assistant_reply})

    print(f"\nAssistant: {assistant_reply[:200]}{'...' if len(assistant_reply) > 200 else ''}")
    print()
    print_usage(response, elapsed)


Turn 1: What is the title of this novel?

Assistant: The title of this novel is **Pride and Prejudice**, written by **Jane Austen**.

  Time:                5.19s
  Input tokens:        3
  Output tokens:       24
  Cache write tokens:  187361

Turn 2: Who are Mr. and Mrs. Bennet?

Assistant: Mr. and Mrs. Bennet are a married couple who are central characters in the novel. They live at **Longbourn** and are the parents of **five daughters**: Jane, Elizabeth, Mary, Catherine (Kitty), and Ly...

  Time:                8.27s
  Input tokens:        3
  Output tokens:       272
  Cache write tokens:  38
  Cache read tokens:   187361

Turn 3: What is Netherfield Park?

Assistant: **Netherfield Park** is a large estate located near the village of **Longbourn** in Hertfordshire, where the Bennet family lives. It plays an important role in the novel as it is the home that is let ...

  Time:                8.74s
  Input tokens:        3
  Output tokens:       300
  Cache write tokens:  283
  C

第一轮之后，后续每一轮几乎 100% 的输入词元都从缓存读取。对话代码只是一个普通的消息列表——各个块上不需要特殊的 `cache_control` 标记。

---
## 示例 3：显式缓存断点

想要更多控制时，可以把 `cache_control` 直接放在各个内容块上。这在以下情形很有用：

- 你希望以不同的 TTL 缓存不同的部分
- 你需要把系统提示词与消息内容分别独立缓存
- 你希望对哪些内容被缓存有精细的控制

你也可以把两种方式结合使用：为系统提示词使用显式断点，同时让自动缓存处理对话。

下面我们把 `cache_control` 直接放在书籍内容块上，并在每一轮手动把断点向前移动。

In [9]:
class ConversationWithExplicitCaching:
    """Multi-turn conversation that manually places cache_control on the last user message."""

    def __init__(self):
        self.turns = []

    def add_user(self, content):
        self.turns.append({"role": "user", "content": [{"type": "text", "text": content}]})

    def add_assistant(self, content):
        self.turns.append({"role": "assistant", "content": [{"type": "text", "text": content}]})

    def get_messages(self):
        """Return messages with cache_control on the last user message."""
        result = []
        last_user_idx = max(i for i, t in enumerate(self.turns) if t["role"] == "user")

        for i, turn in enumerate(self.turns):
            if i == last_user_idx:
                result.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": turn["content"][0]["text"],
                                "cache_control": {"type": "ephemeral"},
                            }
                        ],
                    }
                )
            else:
                result.append(turn)

        return result


conv = ConversationWithExplicitCaching()

for i, question in enumerate(questions, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}: {question}")
    print("=" * 50)

    conv.add_user(question)

    start = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        system=[
            {
                "type": "text",
                "text": system_message,
                "cache_control": {"type": "ephemeral"},  # explicit breakpoint on system
            },
        ],
        messages=conv.get_messages(),
    )
    elapsed = time.time() - start

    assistant_reply = response.content[0].text
    conv.add_assistant(assistant_reply)

    print(f"\nAssistant: {assistant_reply[:200]}{'...' if len(assistant_reply) > 200 else ''}")
    print()
    print_usage(response, elapsed)


Turn 1: What is the title of this novel?

Assistant: The title of this novel is **Pride and Prejudice**, written by **Jane Austen**.

  Time:                4.53s
  Input tokens:        3
  Output tokens:       24
  Cache read tokens:   187361

Turn 2: Who are Mr. and Mrs. Bennet?

Assistant: Mr. and Mrs. Bennet are a married couple who are central characters in the novel. They live at **Longbourn** and are the parents of **five daughters**: Jane, Elizabeth (Lizzy), Mary, Catherine (Kitty)...

  Time:                7.57s
  Input tokens:        3
  Output tokens:       283
  Cache read tokens:   187399

Turn 3: What is Netherfield Park?

Assistant: **Netherfield Park** is a large estate located near the village of **Longbourn** in Hertfordshire, where the Bennet family lives. It plays an important role in the novel as it is the residence that se...

  Time:                6.85s
  Input tokens:        3
  Output tokens:       300
  Cache write tokens:  294
  Cache read tokens:   187399

---
## 如何选择方式

| | 自动缓存 | 显式断点 |
|---|---|---|
| **易用性** | 一行改动 | 必须放置并移动 `cache_control` 断点 |
| **多轮对话** | 断点自动向前移动 | 由你管理断点位置 |
| **精细控制** | 否 | 最多 4 个独立断点 |
| **混合 TTL** | 自动断点只能使用单个 TTL | 每个断点可用不同 TTL |
| **可组合** | 可以——自动与显式同时使用 | 可以 |

**从自动缓存开始。**它以最小的工作量覆盖大多数使用场景。只有在需要精细控制时才改用显式断点。

### 关键细节

- **最小可缓存长度：**Sonnet 为 1,024 个词元；Opus 和 Haiku 4.5 为 4,096 个词元
- **缓存 TTL：**默认 5 分钟（每次命中时刷新）。1 小时 TTL 按基础输入价格的 2 倍计费。
- **价格：**缓存写入按基础输入价格的 1.25 倍计费，缓存读取按基础输入价格的 0.1 倍计费。
- **断点上限：**每个请求最多 4 个显式断点。自动缓存占用一个名额。

完整细节请参阅[提示词缓存文档](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)。